# Dataset Viewer

Browse generated character images grouped by font.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

In [ ]:
PARQUET_PATH = Path("../output/dataset.parquet")
IMAGE_DIR    = Path("../output/images")

df = pd.read_parquet(PARQUET_PATH)
df["font_stem"] = df["font_path"].apply(lambda p: Path(p).stem)
print(f"{len(df):,} records | {df['font_stem'].nunique()} fonts | {df['char'].nunique()} unique chars")
df.head(3)

In [ ]:
def show_font(font_stem: str, cols: int = 16, cell_px: int = 48) -> None:
    """Render all characters for one font as a grid."""
    rows_df = df[df["font_stem"] == font_stem].sort_values("codepoint")
    files = [IMAGE_DIR / r["file"] for _, r in rows_df.iterrows()]
    chars = rows_df["char"].tolist()
    n = len(files)
    if n == 0:
        print(f"No images for {font_stem}")
        return

    n_rows = (n + cols - 1) // cols
    fig_w = cols * cell_px / 72
    fig_h = n_rows * (cell_px + 12) / 72  # +12px for char label
    fig, axes = plt.subplots(n_rows, cols, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(-1)  # flatten regardless of shape

    for ax, fpath, ch in zip(axes, files, chars):
        img = Image.open(fpath)
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(ch, fontsize=6, pad=1)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(font_stem, fontsize=9, y=1.01)
    plt.tight_layout(pad=0.3)
    plt.show()

In [ ]:
# Show all fonts
for font_stem in sorted(df["font_stem"].unique()):
    show_font(font_stem)

In [ ]:
# Show a single font
# show_font("NotoSansJP-Regular")